In [1]:
import numpy as np
from scipy import constants
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker
import itertools

#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
print(omega_tweezer)
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)
#test

3540698434791077.0
[1.47e+08 1.40e+08]


In [ ]:
P_opt = np.linspace(0,10,1) #W
w0 = 1e-6 #m
N = 3 #change me for a different number of ions

In [ ]:
pot = potential(omega_tweezer,linewidths,omega_res,P_opt,w0)
w_tw_r = omega_tweezer_r(pot,w0,m) 
print("f_tw_r = ",w_tw_r/(2*pi*1e3),"kHz")

In [ ]:
f_rf_r = 1e6 #Hz
w_rf_r = f_rf_r*2*pi
ueq = ion_spacing(N,f_rf_r)[0]
f_rf_r_list = np.full(N,f_rf_r)
w_rf_r_list = np.full(N,w_rf_r)


In [ ]:
def lamb_dicke(qubit_wavelength,omega_rf):
    """Inputs:
        qubit wavelength -- float of qubit wavelength, ex 729e-9 for Ca 40
        omega_rf -- secular frequency of trap in whatever direction you're considering (2*pi*Hz)

        This function assumes the incoming beam has a 100% projection on the direction you're considering. 
        If you want what the lamb dicke parameter would be for an ion in a multi-ion chain then you have to scale it by its participation
        Outputs:
        Lamb -Dicke parameter for a single ion
    """


    k = (2*pi)/qubit_wavelength
    np.sqrt((hbar*k**2) /(2*m*omega_rf) )

In [ ]:
def tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1
):
    """
    Compute all tweezer combinations and corresponding radial mode frequencies,
    including power dependence, tweezed/untweezed mode separation,
    and optionally over multiple ion counts N.

    Parameters
    ----------
    N : int or iterable of int
        Number of ions or list of different ion counts to loop over.
    """

    import numpy as np
    import itertools
    import pandas as pd

    pi = np.pi
    rows = []

    # --- Allow N to be iterable ---
    N_list = np.atleast_1d(N)

    # --- Helper functions ---
    def max_with_index(arr):
        arr = np.ravel(arr)
        idx = np.argmax(arr)
        return arr[idx], idx

    def max_no_tweeze(arr, tweezed, N):
        arr = np.ravel(arr)
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)
        if np.any(mask):
            arr_untweezed = arr[mask]
            ion_untweezed = ion_indices[mask]
            idx_local = np.argmax(arr_untweezed)
            return arr_untweezed[idx_local], ion_untweezed[idx_local]
        else:
            return np.nan, np.nan

    def sum_no_tweeze(arr, tweezed, N):
        arr = np.ravel(arr)
        ion_indices = np.arange(N)
        untweezed_mask = ~np.isin(ion_indices, tweezed)
        if np.any(untweezed_mask):
            return np.sqrt(np.sum(arr[untweezed_mask] ** 2))
        else:
            return np.nan

    # --- Outer loop over N values ---
    for N_val in N_list:
        print(f"\n=== Computing for N = {N_val} ions ===")

        # Generate all tweezer combinations up to max_tweezed
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N_val), r))

        # --- Loop over optical powers ---
        for P in P_opt:
            pot = potential(omega_tweezer, linewidths, omega_res, P, w0)
            w_tw_r = omega_tweezer_r(pot, w0, m)
            f_tw_r = w_tw_r / (2 * pi * 1e3)
            print(f"P = {P:.3f} W → f_tw_r = {f_tw_r:.3f} kHz")

            # RF trap setup
            w_rf_r = f_rf_r * 2 * pi
            ueq = ion_spacing(N_val, f_rf_r)[0]
            w_rf_r_list = np.full(N_val, w_rf_r)

            # --- Loop through all tweezer combinations ---
            for positions in all_combos:
                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in positions else w_rf_r_list[i]
                    for i in range(N_val)
                ])

                # Compute radial modes
                modes = mode_calc_r(m, combo, ueq, N_val)

                # --- Build DataFrame row ---
                row = {
                    "N": N_val,
                    "P_opt (W)": P,
                    "Ions tweezed": positions,
                    "Combined radial frequencies": combo,
                    "radial_modes": modes
                }

                # Add arbitrary number of modes dynamically
                for mode_index, mode_values in enumerate(modes):
                    row[f"Mode{mode_index}"] = np.ravel(mode_values)

                rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
N = 3
P_opt = [1] #W
#print(P_opt)
w0 = 1e-6
f_rf_r = 1e6
data = tweezer_combos_full_radial(omega_tweezer, linewidths, omega_res, m, mode_calc_r, N, f_rf_r, P_opt, w0)


In [ ]:
data

In [ ]:
Untweezed = data[data['Ions tweezed']==()]
Tweeze0 = data[data['Ions tweezed']==(0,)]
Tweeze1 = data[data['Ions tweezed']==(1,)]
Tweeze2 = data[data['Ions tweezed']==(2,)]

In [ ]:
Untweezed


In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode0_sum'], label="Untweezed")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode0_sum'], label="Tweeze 0")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode0_sum'], label="Tweeze 1")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode0_sum'], label="Tweeze 2")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode0_sum'], label="Tweeze 0,1")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode0_sum'], label="Tweeze 1,2")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode0_sum'], label="Tweeze 0,2")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode0_sum")
plt.title("Mode0 Sum vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode0_sum_no_tweeze'], label="Untweezed")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode0_sum_no_tweeze'], label="Tweeze 0")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode0_sum_no_tweeze'], label="Tweeze 1")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode0_sum_no_tweeze'], label="Tweeze 2")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode0_sum_no_tweeze'], label="Tweeze 0,1")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode0_sum_no_tweeze'], label="Tweeze 1,2")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode0_sum_no_tweeze'], label="Tweeze 0,2")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode0_sum (excluding tweezed ion)")
plt.title("Mode0 Sum (excluding tweezed ion) vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode0_max'], label=f"Untweezed, ion #{Untweezed['Mode0_max_index'].iloc[2]}")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode0_max'],label = f"Tweeze0, ion #{Tweeze0['Mode0_max_index'].iloc[2]}")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode0_max'], label = f"Tweeze1, ion #{Tweeze1['Mode0_max_index'].iloc[2]}")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode0_max'], label = f"Tweeze2, ion #{Tweeze2['Mode0_max_index'].iloc[2]}")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode0_max'], label = f"Tweeze01, ion #{Tweeze01['Mode0_max_index'].iloc[2]}")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode0_max'], label = f"Tweeze12, ion #{Tweeze12['Mode0_max_index'].iloc[2]}")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode0_max'], label = f"Tweeze02, ion #{Tweeze02['Mode0_max_index'].iloc[2]}")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode0 max vector comp")
plt.title("Mode0 Max vector comp vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode0_max_no_tweeze'], label=f"Untweezed, ion #{Untweezed['Mode0_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode0_max_no_tweeze'],label = f"Tweeze0, ion #{Tweeze0['Mode0_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode0_max_no_tweeze'], label = f"Tweeze1, ion #{Tweeze1['Mode0_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode0_max_no_tweeze'], label = f"Tweeze2, ion #{Tweeze2['Mode0_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode0_max_no_tweeze'], label = f"Tweeze01, ion #{Tweeze01['Mode0_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode0_max_no_tweeze'], label = f"Tweeze12, ion #{Tweeze12['Mode0_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode0_max_no_tweeze'], label = f"Tweeze02, ion #{Tweeze02['Mode0_max_no_tweeze_index'].iloc[2]}")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode0 max vector comp (ignoring tweeze ion)")
plt.title("Mode0 Max vector comp (ignoring tweezed ion) vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode1_sum'], label="Untweezed")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode1_sum'], label="Tweeze 0")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode1_sum'], label="Tweeze 1")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode1_sum'], label="Tweeze 2")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode1_sum'], label="Tweeze 0,1")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode1_sum'], label="Tweeze 1,2")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode1_sum'], label="Tweeze 0,2")

plt.xlabel("P_opt (W)")
plt.ylabel("Mode1_sum")
plt.title("Mode1 Sum vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode1_sum_no_tweeze'], label="Untweezed")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode1_sum_no_tweeze'], label="Tweeze 0")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode1_sum_no_tweeze'], label="Tweeze 1")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode1_sum_no_tweeze'], label="Tweeze 2")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode1_sum_no_tweeze'], label="Tweeze 0,1")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode1_sum_no_tweeze'], label="Tweeze 1,2")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode1_sum_no_tweeze'], label="Tweeze 0,2")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode0_sum (excluding tweezed ion)")
plt.title("Mode0 Sum (excluding tweezed ion) vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode1_max'], label = f"Untweezed, ion #{Untweezed['Mode1_max_index'].iloc[2]}")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode1_max'],  label = f"Tweeze0, ion #{Tweeze0['Mode1_max_index'].iloc[2]}")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode1_max'],  label = f"Tweeze1, ion #{Tweeze1['Mode1_max_index'].iloc[2]}")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode1_max'],  label = f"Tweeze2, ion #{Tweeze1['Mode1_max_index'].iloc[2]}")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode1_max'],  label = f"Tweeze01, ion #{Tweeze01['Mode1_max_index'].iloc[2]}")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode1_max'],  label = f"Tweeze12, ion #{Tweeze12['Mode1_max_index'].iloc[2]}")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode1_max'],  label = f"Tweeze02, ion #{Tweeze02['Mode1_max_index'].iloc[2]}")

plt.xlabel("P_opt (W)")
plt.ylabel("Mode1_Max")
plt.title("Mode1 Max vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode1_max_no_tweeze'], label=f"Untweezed, ion #{Untweezed['Mode1_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode1_max_no_tweeze'],label = f"Tweeze0, ion #{Tweeze0['Mode1_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode1_max_no_tweeze'], label = f"Tweeze1, ion #{Tweeze1['Mode1_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode1_max_no_tweeze'], label = f"Tweeze2, ion #{Tweeze2['Mode1_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode1_max_no_tweeze'], label = f"Tweeze01, ion #{Tweeze01['Mode1_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode1_max_no_tweeze'], label = f"Tweeze12, ion #{Tweeze12['Mode1_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode1_max_no_tweeze'], label = f"Tweeze02, ion #{Tweeze02['Mode1_max_no_tweeze_index'].iloc[2]}")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode1 max vector comp (ignoring tweeze ion)")
plt.title("Mode1 Max vector comp (ignoring tweezed ion) vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode2_sum'], label="Untweezed")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode2_sum'], label="Tweeze 0")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode2_sum'], label="Tweeze 1")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode2_sum'], label="Tweeze 2")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode2_sum'], label="Tweeze 0,1")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode2_sum'], label="Tweeze 1,2")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode2_sum'], label="Tweeze 0,2")

plt.xlabel("P_opt (W)")
plt.ylabel("Mode2_sum")
plt.title("Mode2 Sum vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode2_sum_no_tweeze'], label="Untweezed")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode2_sum_no_tweeze'], label="Tweeze 0")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode2_sum_no_tweeze'], label="Tweeze 1")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode2_sum_no_tweeze'], label="Tweeze 2")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode2_sum_no_tweeze'], label="Tweeze 0,1")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode2_sum_no_tweeze'], label="Tweeze 1,2")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode2_sum_no_tweeze'], label="Tweeze 0,2")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode2_sum (excluding tweezed ion)")
plt.title("Mode2 Sum (excluding tweezed ion) vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode2_max'], label = f"Untweezed, ion #{Untweezed['Mode2_max_index'].iloc[2]}")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode2_max'], label = f"Tweeze0, ion #{Tweeze0['Mode2_max_index'].iloc[2]}")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode2_max'],label = f"Tweeze1, ion #{Tweeze1['Mode2_max_index'].iloc[2]}")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode2_max'], label = f"Tweeze2, ion #{Tweeze2['Mode2_max_index'].iloc[2]}")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode2_max'], label = f"Tweeze01, ion #{Tweeze01['Mode2_max_index'].iloc[2]}")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode2_max'], label = f"Tweeze12, ion #{Tweeze12['Mode2_max_index'].iloc[2]}")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode2_max'], label = f"Tweeze02, ion #{Tweeze02['Mode2_max_index'].iloc[2]}")

plt.xlabel("P_opt (W)")
plt.ylabel("Mode2_max")
plt.title("Mode2 max vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(Untweezed['P_opt (W)'], Untweezed['Mode2_max_no_tweeze'], label=f"Untweezed, ion #{Untweezed['Mode2_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze0['P_opt (W)'], Tweeze0['Mode2_max_no_tweeze'],label = f"Tweeze0, ion #{Tweeze0['Mode2_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze1['P_opt (W)'], Tweeze1['Mode2_max_no_tweeze'], label = f"Tweeze1, ion #{Tweeze1['Mode2_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze2['P_opt (W)'], Tweeze2['Mode2_max_no_tweeze'], label = f"Tweeze2, ion #{Tweeze2['Mode2_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze01['P_opt (W)'], Tweeze01['Mode2_max_no_tweeze'], label = f"Tweeze01, ion #{Tweeze01['Mode2_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze12['P_opt (W)'], Tweeze12['Mode2_max_no_tweeze'], label = f"Tweeze12, ion #{Tweeze12['Mode2_max_no_tweeze_index'].iloc[2]}")
plt.scatter(Tweeze02['P_opt (W)'], Tweeze02['Mode2_max_no_tweeze'], label = f"Tweeze02, ion #{Tweeze02['Mode2_max_no_tweeze_index'].iloc[2]}")


plt.xlabel("P_opt (W)")
plt.ylabel("Mode2 max vector comp (ignoring tweeze ion)")
plt.title("Mode2 Max vector comp (ignoring tweezed ion) vs Tweezer Power")
plt.legend(loc = "best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
N = 10
P_opt = np.linspace(0.1,0.5,2) #W
#print(P_opt)
w0 = 1e-6
f_rf_r = 1e6
dataN10 = tweezer_combos_full_radial(omega_tweezer, linewidths, omega_res, m, mode_calc_r, N, f_rf_r, P_opt, w0,max_tweezed=4)

In [ ]:
dataN10

In [ ]:
def split_by_tweeze(data):
    """
    Split a DataFrame into sub-DataFrames by unique 'Ions tweezed' combinations.
    
    Parameters
    ----------
    data : pandas.DataFrame
        The DataFrame with a column 'Ions tweezed' containing tuples.
    
    Returns
    -------
    dict
        Dictionary mapping a readable label (e.g. 'Tweeze01') to sub-DataFrame.
    """
    split_data = {}
    for combo in data['Ions tweezed'].unique():
        if combo == ():
            name = "Untweezed"
        else:
            name = "Tweeze" + "".join(map(str, combo))
        split_data[name] = data[data['Ions tweezed'] == combo]
    return split_data

def split_by_mode(data, N):
    """
    Split a DataFrame into N sub-DataFrames, each including all rows and all
    shared (non-mode) columns, but only the mode-specific columns for that mode.
    """
    split_modes = {}

    # Identify shared columns (not specific to any mode)
    shared_cols = [c for c in data.columns if not c.startswith("Mode")]

    for i in range(N):
        mode = f"Mode{i}"
        mode_cols = [c for c in data.columns if c.startswith(mode)]
        cols = shared_cols + mode_cols
        split_modes[mode] = data[cols].copy()

    return split_modes


In [ ]:
split_by_tweezers_N10 = split_by_tweeze(dataN10)
split_by_modes_N10 = split_by_mode(dataN10,N)


In [ ]:
split_by_modes_N10["Mode0"]

In [ ]:
split_by_modes_N10["Mode9"]

# Now I want to sweep over N ions, limit the maximum number of tweezer beams to 2, and fix one cooling ion

I know the cooling time per mode is inversely proportional to the max of lamb-dicke parameter of mode (max meaning I'm picking out which ion in each mode has the maximum lamb-dicke)

I'm assuming that the k-vector part of the lamb-dicke parameter is equal per ion, so I'm only going to consider the mode-vector coupling term for now

This means I'll pick out the mode-vector coupling term per mode to find the time per mode

Then I'll sum up all of those times per mode to get the total time

In [ ]:
def tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
    qubit_wavelength=729e-9,
    theta=0
):
    """
    Compute all tweezer combinations and corresponding radial mode frequencies,
    including power dependence and tweezed/untweezed mode separation.

    mode_calc_r is expected to return a list of tuples:
        [(freq1, eigvec1), (freq2, eigvec2), ...]
    """

    import numpy as np
    import itertools
    import pandas as pd

    # --- Normalize inputs ---
    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []

    # --- Helper functions ---
    def max_eig_index(eigvec):
        """Return max amplitude and its ion index"""
        idx = np.argmax(np.abs(eigvec))
        return eigvec[idx], idx

    def max_eig_index_no_tweeze(eigvec, tweezed):
        N = len(eigvec)
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)  # untweezed ions only
        if np.any(mask):
            eig_untweezed = eigvec[mask]
            ions_untweezed = ion_indices[mask]
            idx_local = np.argmax(np.abs(eig_untweezed))
            return eig_untweezed[idx_local], ions_untweezed[idx_local]
        else:
            return np.nan, np.nan

    def sum_no_tweeze(arr, tweezed, N):
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)
        if np.any(mask):
            return np.sqrt(np.sum(arr[mask] ** 2))
        else:
            return np.nan

    # --- Loop over number of ions ---
    for N in N_list:
        # Generate all possible tweezer combinations
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N), r))

        # RF trap setup
        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_r)[0]

        # --- Loop over optical powers ---
        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                # Compute tweezer potential for this configuration
                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                # Combine tweezed and untweezed radial frequencies
                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                # --- Compute radial modes ---
                modes = mode_calc_r(m, combo, ueq, N)

                # Extract frequencies and eigenvectors
                freqs = np.array([f for f, v in modes], dtype=float)
                eigvecs = np.vstack([np.ravel(v) for f, v in modes])  # shape (n_modes, N)

                # --- Initialize row ---
                row = {
                    "N": N,
                    #"P_total (W)": P_total,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    #"Combined radial frequencies": combo,
                }

                # --- Loop over modes dynamically ---
                for mode_index, (f, eigvec) in enumerate(modes):
                    eigvec = np.ravel(eigvec)
                    mode_sum_no_tweeze = sum_no_tweeze(eigvec, tweezed_positions, N)
                    mode_max, mode_max_index = max_eig_index(eigvec)
                    mode_max_no_tweeze, mode_max_no_tweeze_index = max_eig_index_no_tweeze(eigvec, tweezed_positions)

                    # Store per-mode data
                    row[f"Mode{mode_index}_freq"] = f
                    row[f"Mode{mode_index}_eigvec"] = eigvec
                    #row[f"Mode{mode_index}_max"] = mode_max
                    #row[f"Mode{mode_index}_max_index"] = mode_max_index
                    #row[f"Mode{mode_index}_max_no_tweeze"] = mode_max_no_tweeze
                    #row[f"Mode{mode_index}_max_no_tweeze_index"] = mode_max_no_tweeze_index

                # --- Unique mode→ion mapping (by absolute value, preserving sign) ---
                abs_eigs = np.abs(eigvecs)
                used = set()
                mode_to_ion_indices = []
                mode_to_ion_amplitudes = []

                for mode_i in range(abs_eigs.shape[0]):
                    order = np.argsort(abs_eigs[mode_i])[::-1]
                    for idx in order:
                        if idx not in used:
                            # pick by largestx |amplitude| but store signed value
                            mode_to_ion_indices.append(int(idx))
                            mode_to_ion_amplitudes.append(float(eigvecs[mode_i, idx]))
                            used.add(idx)
                            break

                row["Mode→Ion indices"] = mode_to_ion_indices
                row["Mode→Ion amplitudes"] = mode_to_ion_amplitudes

                # --- Maximum (by |amplitude|) of assigned amplitudes ---
                row["Max amplitude (mode→ion)"] = (
                    max(map(abs, mode_to_ion_amplitudes)) if mode_to_ion_amplitudes else np.nan
                )

                # --- Store completed row ---
                rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
N = np.arange(3,4)
P_opt = [1]
data= tweezer_combos_full_radial(omega_tweezer,linewidths,omega_res,m,mode_calc_r,N,f_rf_r,P_opt,w0)

In [ ]:
data

In [ ]:
def compute_mode_coupling_row(row, qubit_wavelength=729e-9, theta_deg=0):
    amps = np.atleast_1d(row.get('Mode→Ion amplitudes', []))
    # gather matching Mode{i}_freq entries (0 -> Mode0_freq, 1 -> Mode1_freq, ...)
    freqs = []
    for i in range(len(amps)):
        freqs.append(row.get(f"Mode{i}_freq", np.nan))
    amps = np.array(amps, dtype=float)
    freqs = np.array(freqs, dtype=float)

    # if any freq missing return array of nans with same length as amps
    if amps.size == 0 or np.any(np.isnan(freqs)):
        return np.full(amps.shape, np.nan, dtype=float)

    # prefactor and corrected sqrt (sqrt(hbar / (2*m*omega)))
    prefactor = (2 * np.pi * qubit_wavelength / c) * np.cos(np.radians(theta_deg))
    coupling = amps * prefactor * np.sqrt(hbar / (2 * m * freqs))
    return coupling

# create column with per-row arrays
data['Mode→Ion_coupling'] = data.apply(_compute_mode_coupling_row, axis=1)


In [ ]:
compute_mode_coupling_row()
data['Mode→Ion_coupling'] = data.apply(compute_mode_coupling_row, axis=1)


In [ ]:
#next step: take the list of amplitudes and convert it to times, plot the minimum time per N and sweep over N

# look at this one later, it doubles up on indices for a tweezed ion

In [ ]:
# ...existing code...
def tweezer_combos_full_radial_with_mode_mapping_ignore_tweezed(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
    qubit_wavelength=729e-9,
    theta=0,
):
    """
    Computes radial modes and eigenvectors and tracks:
    - Max per mode and no-tweeze max
    - Mode→Ion mapping ignoring tweezed ions (duplicates allowed)
    - Mode→Ion amplitudes and max amplitude
    - Mode→Ion couplings (per-row array and per-mode Mode{i}_coupling columns)
    """

    import numpy as np
    import itertools
    import pandas as pd
    from scipy import constants as _const

    c = _const.c
    hbar = _const.hbar

    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []

    def max_eig_index(eigvec):
        idx = np.argmax(np.abs(eigvec))
        return eigvec[idx], idx

    def max_eig_index_no_tweeze(eigvec, tweezed):
        N = len(eigvec)
        ion_indices = np.arange(N)
        mask = ~np.isin(ion_indices, tweezed)
        if np.any(mask):
            eig_untweezed = eigvec[mask]
            ions_untweezed = ion_indices[mask]
            idx_local = np.argmax(np.abs(eig_untweezed))
            return eig_untweezed[idx_local], ions_untweezed[idx_local]
        else:
            return np.nan, np.nan

    for N in N_list:
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N), r))

        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_r)[0]

        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                modes = mode_calc_r(m, combo, ueq, N)

                # frequencies and eigenvector matrix (n_modes, N)
                freqs = np.array([f for f, v in modes], dtype=float)
                eigvecs = np.vstack([np.ravel(v) for f, v in modes]) if len(modes) else np.empty((0, N))

                row = {
                    "N": N,
                    "P_total (W)": P_total,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    "Combined radial frequencies": combo,
                }

                mode_to_ion_indices = []
                mode_to_ion_amplitudes = []

                for mode_index, (f, eigvec) in enumerate(modes):
                    eigvec = np.ravel(eigvec)
                    mode_max, mode_max_index = max_eig_index(eigvec)
                    mode_max_no_tweeze, mode_max_no_tweeze_index = max_eig_index_no_tweeze(eigvec, tweezed_positions)

                    row[f"Mode{mode_index}_max"] = mode_max
                    row[f"Mode{mode_index}_max_index"] = mode_max_index
                    row[f"Mode{mode_index}_max_no_tweeze"] = mode_max_no_tweeze
                    row[f"Mode{mode_index}_max_no_tweeze_index"] = mode_max_no_tweeze_index

                    # --- Mode→Ion mapping ignoring tweezed ions ---
                    candidates = [i for i in range(N) if i not in tweezed_positions]
                    if not candidates:
                        mode_to_ion_indices.append(np.nan)
                        mode_to_ion_amplitudes.append(np.nan)
                    else:
                        max_idx_local = candidates[np.argmax(np.abs(eigvec[candidates]))]
                        mode_to_ion_indices.append(int(max_idx_local))
                        mode_to_ion_amplitudes.append(float(eigvec[max_idx_local]))

                row["Mode→Ion indices"] = mode_to_ion_indices
                row["Mode→Ion amplitudes"] = mode_to_ion_amplitudes
                row["Max amplitude (mode→ion)"] = max(
                    map(abs, [x for x in mode_to_ion_amplitudes if not np.isnan(x)])
                ) if any(not np.isnan(x) for x in mode_to_ion_amplitudes) else np.nan

                # --- Compute mode→ion coupling (per-row array + per-mode columns) ---
                k = 2 * np.pi / qubit_wavelength
                proj = np.cos(np.radians(theta))
                couplings = []
                for i, amp in enumerate(mode_to_ion_amplitudes):
                    # amplitude may be nan
                    try:
                        amp_f = float(amp)
                    except Exception:
                        amp_f = np.nan

                    freq_i = freqs[i] if (i < len(freqs)) else np.nan

                    if np.isnan(amp_f) or np.isnan(freq_i) or freq_i == 0:
                        val = np.nan
                    else:
                        # use sqrt(hbar/(2*m*omega)) where freqs are angular frequencies expected from mode_calc_r
                        val = amp_f * k * proj * np.sqrt(hbar / (2 * m * freq_i))

                    couplings.append(float(val) if not np.isnan(val) else np.nan)
                    row[f"Mode{i}_coupling"] = float(val) if not np.isnan(val) else np.nan

                row["Mode→Ion_coupling"] = couplings

                rows.append(row)

    return pd.DataFrame(rows)
# ...existing code...


In [ ]:
N = np.arange(3,4)
P_opt = [100e-3]
data= tweezer_combos_full_radial_with_mode_mapping_ignore_tweezed(omega_tweezer,linewidths,omega_res,m,mode_calc_r,N,f_rf_r,P_opt,w0)

In [ ]:
data

In [ ]:
#what is the two photon rabi frequency for each of these?????
#effective rabi frequency, g = eta*Omega/2 from Sideband Thermometry of Ion Crystals -- PRX Quantum published in 2023
#and regular Rabi frequency is= 
# Rabi_squared = [(linewidths[0]*6*pi*c**2)/(hbar*omega_res[0]**3) * I , linewidths[1] *(6*pi*c**2)/(hbar*omega_res[1]**3) * I]
#but i need the linewidths and omegares for the qubit transition 
#or refer to equation 2.17 from the Hempel thesis
#also, use I = 2P/(pi w0^2)

1. find the saturation intensity of the qubit transition
2. insert the linewidth and resonant frequency of the qubit transition
3. find the rabi frequency with the equation I used above
4. take my mode vector contribution values and convert them to lamb-dicke factors
5. find the effective rabi frequency

linewidth_729 = 136e-3*2*pi
omega_res_729 = 

# brute force calculation of every single ion coupling to every single mode -- radial only

In [2]:
# ...existing code...
def tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N_list,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
):
    """
    Compute all tweezer combinations and corresponding radial mode frequencies,
    including power dependence and tweezed/untweezed mode separation.

    mode_calc_r is expected to return a list of tuples:
        [(freq1, eigvec1), (freq2, eigvec2), ...]
    This version ensures the dataframe has Mode0_eigvec, Mode1_eigvec, ... Mode{N-1}_eigvec
    and Mode0_freq, Mode1_freq, ... Mode{N-1}_freq for each row (missing entries filled with NaN).
    """

    import numpy as np
    import itertools
    import pandas as pd

    # --- Normalize inputs ---
    if np.isscalar(N_list):
        N_list = [int(N_list)]
    if np.isscalar(P_opt):
        P_opt = [P_opt]

    pi = np.pi
    rows = []

    # --- Loop over number of ions ---
    for N in N_list:
        # Generate all possible tweezer combinations
        all_combos = []
        for r in range(0, max_tweezed + 1):
            all_combos.extend(itertools.combinations(range(N), r))

        # RF trap setup
        w_rf_r = f_rf_r * 2 * pi
        w_rf_r_list = np.full(N, w_rf_r)
        ueq = ion_spacing(N, f_rf_r)[0]

        # --- Loop over optical powers ---
        for P_total in P_opt:
            for tweezed_positions in all_combos:
                n_tweezed = len(tweezed_positions)
                P_per = P_total / n_tweezed if n_tweezed > 0 else 0.0

                # Compute tweezer potential for this configuration
                pot = potential(omega_tweezer, linewidths, omega_res, P_per, w0)
                w_tw_r = omega_tweezer_r(pot, w0, m)

                # Combine tweezed and untweezed radial frequencies
                combo = np.array([
                    np.sqrt(w_tw_r**2 + w_rf_r_list[i]**2) if i in tweezed_positions else w_rf_r_list[i]
                    for i in range(N)
                ])

                # --- Compute radial modes ---
                modes = mode_calc_r(m, combo, ueq, N)

                # Extract frequencies and eigenvectors
                freqs = np.array([f for f, v in modes], dtype=float) if len(modes) else np.array([], dtype=float)
                if len(modes):
                    eigvecs = np.vstack([np.ravel(v) for f, v in modes])  # shape (n_modes, N)
                else:
                    eigvecs = np.empty((0, N))

                # --- Initialize row with shared info ---
                row = {
                    "N": N,
                    "Tweezed ions": tweezed_positions,
                    "P_per_tweezer (W)": P_per,
                    "Combined radial frequencies": combo,
                }

                # --- Ensure columns for all possible modes up to N exist per row ---
                # Fill Mode{i}_freq and Mode{i}_eigvec for i in [0, N-1]
                for mode_index in range(N):
                    # frequency
                    if mode_index < len(freqs):
                        row[f"Mode{mode_index}_freq"] = float(freqs[mode_index])
                    else:
                        row[f"Mode{mode_index}_freq"] = np.nan

                    # eigenvector (length N) or NaN array
                    if mode_index < eigvecs.shape[0]:
                        row[f"Mode{mode_index}_eigvec"] = np.ravel(eigvecs[mode_index]).astype(float)
                    else:
                        # use full-length nan array to keep shape consistent
                        row[f"Mode{mode_index}_eigvec"] = np.full(N, np.nan, dtype=float)

                # --- Store completed row ---
                rows.append(row)

    return pd.DataFrame(rows)


In [4]:
N = np.arange(3,4)
P_opt = [100e-3]
w0 = 1e-6
f_rf_r = 1e6
N3= tweezer_combos_full_radial(
    omega_tweezer,
    linewidths,
    omega_res,
    m,
    mode_calc_r,
    N,
    f_rf_r,
    P_opt,
    w0,
    max_tweezed=1,
)
N3

,N,Tweezed ions,P_per_tweezer (W),Combined radial frequencies,Mode0_freq,Mode0_eigvec,Mode1_freq,Mode1_eigvec,Mode2_freq,Mode2_eigvec
0,3,(),0.0,"[6283185.307179586, 6283185.307179586, 6283185...",1.000000e+06,"[0.577350269189621, 0.5773502691896278, 0.5773...",987253.616904,"[0.7071067811865754, -6.038822267752199e-14, -...",969127.076195,"[-0.4082482904638137, 0.8164965809277251, -0.4..."
1,3,"(0,)",0.1,"[6595799.989691874, 6283185.307179586, 6283185...",1.040601e+06,"[-0.9836895964499031, -0.17081996742552139, -0...",994305.089283,"[-0.135804001686669, 0.4998713272350667, 0.855...",971785.786802,"[0.11794935761636946, -0.849087271684716, 0.51..."
2,3,"(1,)",0.1,"[6283185.307179586, 6595799.989691874, 6283185...",1.034651e+06,"[0.21289466185572722, 0.9535993529290289, 0.21...",987253.616904,"[-0.7071067811869692, 1.7583265342019252e-13, ...",985234.808714,"[0.6742965689907784, -0.301078518153219, 0.674..."
3,3,"(2,)",0.1,"[6283185.307179586, 6283185.307179586, 6595799...",1.040601e+06,"[0.056349947337785096, 0.17081996742554714, 0....",994305.089283,"[-0.8553864210602142, -0.49987132723500277, 0....",971785.786802,"[0.5149162593090536, -0.8490872716847487, 0.11..."


In [5]:
Mode0s

NameError: name 'Mode0s' is not defined

In [ ]:

for i in range(len(N3)):
    # name first iteration specially, then use zero-based suffixes for remaining
    if i == 0:
        name = "combinations_tw_None"
    else:
        name = f"combinations_tw{i-1}"
    combined = list(itertools.product(Mode0s.iloc[i], Mode1s.iloc[i], Mode2s.iloc[i]))
    globals()[name] = combined
    print(f"{name}: {len(combined)} items")

def lowest_lists_by_min(data):
    nonempty = [lst for lst in data if lst]
    if not nonempty:
        return []
    # compute the minimum of |values| in each list
    lowest_abs_min = min(min(abs(x) for x in lst) for lst in nonempty)
    # return lists whose abs-min equals that
    return [lst for lst in nonempty if min(abs(x) for x in lst) == lowest_abs_min]

def highest_lists_by_max(data):
    nonempty = [lst for lst in data if lst]
    if not nonempty:
        return []
    # compute the maximum of |values| in each list
    highest_abs_max = max(max(abs(x) for x in lst) for lst in nonempty)
    # return lists whose abs-max equals that
    return [lst for lst in nonempty if max(abs(x) for x in lst) == highest_abs_max]



combinations_tw_None: 27 items
combinations_tw0: 27 items
combinations_tw1: 27 items
combinations_tw2: 27 items


In [43]:
filter1_tw0 = lowest_lists_by_min(combinations_tw0)
filter1_tw1 = lowest_lists_by_min(combinations_tw1)
filter1_tw2 = lowest_lists_by_min(combinations_tw2)
print(filter1_tw0)
print(len(filter1_tw0))
print(filter1_tw1)
print(len(filter1_tw1))
print(filter1_tw2)
print(len(filter1_tw2))

filter2_tw0 = highest_lists_by_max(filter1_tw0)
filter2_tw1 = highest_lists_by_max(filter1_tw1)
filter2_tw2 = highest_lists_by_max(filter1_tw2)





[(-0.05634994733778222, -0.135804001686669, 0.11794935761636946), (-0.05634994733778222, -0.135804001686669, -0.849087271684716), (-0.05634994733778222, -0.135804001686669, 0.5149162593091144), (-0.05634994733778222, 0.4998713272350667, 0.11794935761636946), (-0.05634994733778222, 0.4998713272350667, -0.849087271684716), (-0.05634994733778222, 0.4998713272350667, 0.5149162593091144), (-0.05634994733778222, 0.855386421060178, 0.11794935761636946), (-0.05634994733778222, 0.855386421060178, -0.849087271684716), (-0.05634994733778222, 0.855386421060178, 0.5149162593091144)]
9
[(0.21289466185572722, 1.7583265342019252e-13, 0.6742965689907784), (0.21289466185572722, 1.7583265342019252e-13, -0.301078518153219), (0.21289466185572722, 1.7583265342019252e-13, 0.6742965689916613), (0.9535993529290289, 1.7583265342019252e-13, 0.6742965689907784), (0.9535993529290289, 1.7583265342019252e-13, -0.301078518153219), (0.9535993529290289, 1.7583265342019252e-13, 0.6742965689916613), (0.2128946618557476, 

In [44]:
print(filter2_tw0)
print(len(filter2_tw0))
print(filter2_tw1)
print(len(filter2_tw1))
print(filter2_tw2)
print(len(filter2_tw2))

[(-0.05634994733778222, 0.855386421060178, 0.11794935761636946), (-0.05634994733778222, 0.855386421060178, -0.849087271684716), (-0.05634994733778222, 0.855386421060178, 0.5149162593091144)]
3
[(0.9535993529290289, 1.7583265342019252e-13, 0.6742965689907784), (0.9535993529290289, 1.7583265342019252e-13, -0.301078518153219), (0.9535993529290289, 1.7583265342019252e-13, 0.6742965689916613)]
3
[(0.056349947337785096, -0.8553864210602142, 0.5149162593090536), (0.056349947337785096, -0.8553864210602142, -0.8490872716847487), (0.056349947337785096, -0.8553864210602142, 0.11794935761639991)]
3
